In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated_pipeline/1_setup/utils

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","gross_price","Data Source")



In [0]:
catalog=dbutils.widgets.get('catalog')
data_source=dbutils.widgets.get('data_source')

base_path=f"s3://sportsbar-aws-dp/{data_source}/*.csv"


##bronze processing

In [0]:
df_bronze=spark.read.format("csv").option("inferSchema","true").option("header","true").load(base_path)\
    .withColumn("read_timestmap",F.current_timestamp()).select("*","_metadata.file_name","_metadata.file_size")

In [0]:
df_bronze.write.format("delta").mode("overwrite").option("delta.enableChangeDataFeed","true").saveAsTable(f"{catalog}\
    .{bronze_schema}.{data_source}")

##silver

In [0]:
df_bronze=spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

In [0]:
df_bronze.select("month").distinct().show()

In [0]:
df_silver=df_bronze.withColumn("month",F.coalesce(F.try_to_date(F.col("month"),"dd/MM/yyyy"),\
    F.try_to_date(F.col("month"),"dd-MM-yyyy"),\
    F.try_to_date(F.col("month"),"yyyy/MM/dd"),
    F.try_to_date(F.col("month"),"yyyy-MM-dd")))

In [0]:
df_silver=df_silver.withColumn("gross_price",F.when(F.col("gross_price").cast("string").rlike("^[+-]?[0-9]+$"),\
    F.when(F.col("gross_price").cast("double")<0,F.col("gross_price").cast("double")*-1).otherwise(F.col("gross_price").cast("double"))).otherwise(0))

In [0]:
df_products=spark.table("fmcg.silver.products")
df_joined=df_silver.alias("t").join(df_products.select("product_id","product_code").alias("s"),on=F.col("t.product_id")==F.col("s.product_id")\
    ,how="inner")

df_joined=df_joined.select("t.product_id","product_code","month","gross_price","read_timestmap","file_name","file_size")

In [0]:
df_joined.write.mode("overwrite").format("delta").option("delta.enableChangeDataFeed","true").option("mergeSchema","true")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

##Gold processing


In [0]:
df_silver=spark.sql(f"select product_code,month,gross_price from {catalog}.{silver_schema}.{data_source}")


In [0]:
df_gold_price=df_silver.withColumn("year",F.year(F.col("month"))).withColumn("is_zero",F.when(F.col("gross_price")==0,1).otherwise(0))
from pyspark.sql.window import Window
w=Window.partitionBy("product_code","year").orderBy(F.col("is_zero"),F.col("month").desc())
df_gold_price=df_gold_price.withColumn("rnk",F.row_number().over(w))
df_gold_price=df_gold_price.filter(F.col("rnk")==1)
df_gold_price=df_gold_price.select("product_code","gross_price","year")

df_gold_price.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema","true")\
    .option("delta.enableChangeDataFeed","true")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_{data_source}")
                                                                             

In [0]:
delta_table=DeltaTable.forName(spark,f"{catalog}.{gold_schema}.dim_{data_source}")
df_gold_price=spark.table(f"{catalog}.{gold_schema}.sb_{data_source}")
delta_table.alias('t').merge(source=df_gold_price.alias('s'),condition=F.col('t.product_code')==F.col('s.product_code'))\
    .whenMatchedUpdate(set={'price_inr':"s.gross_price","year":'s.year'}).whenNotMatchedInsert(values={"product_code":"s.product_code","price_inr"\
        :"s.gross_price"}).execute()